# 7 — Projections & sense lineages

Downstream of `6_embedding-classification-applied.ipynb`, which writes
`data/large_files/<term>_embeddigings_full_classified.parquet`: the **full**
concordances labelled by the embedding-based classifier (`class_<model>_embedding_pred`).

The notebook is deliberately gradual:

1. **Data** — load the classified full data (or a sample fallback) and derive dates.
2. **Embeddings** — assemble vectors exactly as the saved classifier saw them.
3. **Diagnostic** — *how much sense structure is there?* (unsupervised vs LDA) so we
   only pay for a supervised projection if we actually need one.
4. **Context maps** — unsupervised 2D (UMAP, PCA+t-SNE).
5. **Sense discrimination** — one supervised LDA view, for comparison.
6. **Diachronic composition** — sense shares across centuries (no embedding needed).
7. **Lineage** — per-sense century centroids → 3D trajectories (time on Z).

Everything is dimension-agnostic: change `EMB_SPEC` to any single column or a list.

## 1. Setup

In [ ]:
import importlib
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Reload helpers after editing them (from x import y binds names, not modules).
import projections_helpers, lda_projections
importlib.reload(projections_helpers)
importlib.reload(lda_projections)
from projections_helpers import build_embedding_matrix, create_label_palette

DATA = Path("../data")
FULL_DIR = DATA / "large_files"
FIGDIR = Path("../figures"); FIGDIR.mkdir(exist_ok=True)

TERMS = ["passio", "pathos"]
SEED = 42
MIN_CENTURY_SAMPLES = 3     # minimum rows for a valid (sense, century) centroid
SAMPLE_CAP = 5000           # subsample cap for expensive 2D methods on full data
print("terms:", TERMS)

## 2. Data

Prefer the applied notebook's **full** output; fall back to the 2000-instance sample
so the notebook still runs before that file exists. Dates (`random_date`) and century
midpoints are derived here — they are not raw columns.

In [ ]:
import tempun


def load_term(term):
    """Full embedding-classified output if present, else the 2000-sample file."""
    full = FULL_DIR / f"{term}_embeddigings_full_classified.parquet"
    sample = FULL_DIR / f"{term}_sample2000_classified_embeddings.parquet"
    if full.is_file():
        return pd.read_parquet(full), "full"
    return pd.read_parquet(sample), "sample2000"


def century_midpoint(year):
    """Year -> century midpoint (BCE-safe); year 0 -> None."""
    if pd.isna(year):
        return None
    year = int(year)
    if year == 0:
        return None
    if year < 0:
        c = (-year + 99) // 100
        return -(c * 100 - 50)
    c = (year - 1) // 100 + 1
    return (c - 1) * 100 + 50


def _random_date(row):
    d = tempun.model_date(row["not_before"], row["not_after"])
    if isinstance(d, (list, tuple, np.ndarray)):
        d = d[0]
    return d


datasets = {}
for term in TERMS:
    df, source = load_term(term)
    # Prediction column from the applied notebook, else the LLM label column.
    pred = [c for c in df.columns if c.endswith("_embedding_pred")]
    label_col = pred[0] if pred else sorted(c for c in df.columns if c.startswith("class_"))[0]
    df = df[df[label_col].notna()].copy()
    if "random_date" not in df.columns:
        df["random_date"] = df.apply(_random_date, axis=1)
    df = df[df["random_date"].notna()].copy()
    df["sense"] = df[label_col].astype(int)
    df["century"] = df["random_date"].map(century_midpoint)
    datasets[term] = {"df": df, "source": source, "label_col": label_col}
    print(f"{term:8s} source={source:11s} rows={len(df):>6,} senses={df['sense'].nunique()} "
          f"label={label_col!r}")

## 3. Embeddings

Use the exact spec the saved classifier was trained on (`embedding_cols` + `merge_method`).

In [ ]:
import joblib


def classifier_spec(term):
    art = DATA / f"{term}_wsd_philberta_l11_mdeberta_l9_concat.joblib"
    if art.is_file():
        a = joblib.load(art)
        return a["embedding_cols"], a["merge_method"], a.get("label_map")
    return ["emb_mdeberta_l9"], "concat", None


specs = {t: classifier_spec(t) for t in TERMS}
EMB_SPEC, EMB_METHOD, _ = specs["passio"]      # edit here to override
print(f"EMB_SPEC={EMB_SPEC}  method={EMB_METHOD}")

for term in TERMS:
    df = datasets[term]["df"]
    X = build_embedding_matrix(df, EMB_SPEC, method=EMB_METHOD)
    datasets[term]["X"] = X
    datasets[term]["y"] = df["sense"].values
    print(f"{term:8s} X={X.shape}")

## 4. Diagnostic — how much sense structure is there?

Before projecting, measure whether senses actually separate. `silhouette` rewards
compact, well-separated clusters; `kNN purity` (fraction of the 10 nearest neighbours
sharing the label) detects *local* structure even when clusters overlap. Compare both
against the **chance** kNN purity. If unsupervised methods stay near chance, a
supervised projection (LDA) is doing the real work — treat it accordingly.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.metrics import silhouette_score
from sklearn.neighbors import NearestNeighbors
import umap


def subsample(X, y, cap=SAMPLE_CAP, seed=SEED):
    if len(X) <= cap:
        return X, y, np.arange(len(X))
    idx = np.random.default_rng(seed).choice(len(X), cap, replace=False)
    return X[idx], y[idx], idx


def knn_purity(Z, y, k=10):
    nn = NearestNeighbors(n_neighbors=min(k + 1, len(Z))).fit(Z)
    nbr = nn.kneighbors(return_distance=False)[:, 1:]
    return float(np.mean([np.mean(y[nbr[i]] == y[i]) for i in range(len(y))]))


def chance_purity(y):
    p = np.bincount(y) / len(y)
    return float((p ** 2).sum())


def diagnose(term):
    Xs, ys, _ = subsample(datasets[term]["X"], datasets[term]["y"])
    views = {
        "PCA(2)":      PCA(n_components=2, random_state=SEED).fit_transform(Xs),
        "PCA10+t-SNE": TSNE(2, perplexity=30, init="pca", random_state=SEED)
                         .fit_transform(PCA(n_components=10, random_state=SEED).fit_transform(Xs)),
        "UMAP":        umap.UMAP(n_neighbors=15, min_dist=0.1, metric="cosine",
                                 random_state=SEED).fit_transform(Xs),
        "LDA(2) supervised": LDA(n_components=2, solver="svd").fit_transform(Xs, ys),
    }
    rows = [{"lexeme": term, "method": name,
             "silhouette": silhouette_score(Z, ys), "knn_purity": knn_purity(Z, ys)}
            for name, Z in views.items()]
    rows[0]["chance_knn"] = chance_purity(ys)
    return pd.DataFrame(rows)


diagnostics = pd.concat([diagnose(t) for t in TERMS], ignore_index=True)
diagnostics

## 5. Context maps — unsupervised 2D

Coloured by predicted sense. This is the honest, label-free view: it shows how much sense structure the embedding carries on its own.

In [ ]:
def context_maps(term):
    X, y = datasets[term]["X"], datasets[term]["y"]
    Xs, ys, _ = subsample(X, y)
    maps = {
        "UMAP (cosine)": umap.UMAP(n_neighbors=15, min_dist=0.1, metric="cosine",
                                   random_state=SEED).fit_transform(Xs),
        "PCA(10) + t-SNE": TSNE(2, perplexity=30, init="pca", random_state=SEED)
                             .fit_transform(PCA(n_components=10, random_state=SEED).fit_transform(Xs)),
    }
    palette = create_label_palette(len(np.unique(ys)))
    fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
    for ax, (name, Z) in zip(axes, maps.items()):
        ax.scatter(Z[:, 0], Z[:, 1], c=[palette[v] for v in ys],
                   s=12, alpha=0.7, linewidths=0)
        ax.set_title(f"{term} — {name}"); ax.axis("off")
    fig.suptitle(f"{term}: unsupervised context maps (predicted senses)", y=1.0)
    fig.tight_layout()
    fig.savefig(FIGDIR / f"{term}_context_maps.png", dpi=150, bbox_inches="tight")
    plt.show()


for term in TERMS:
    context_maps(term)

## 6. Sense discrimination — supervised LDA

Kept for comparison: LDA *maximises* sense separation, so it is the sharpest view but also the most circular (it uses the labels to choose the axes).

In [ ]:
from lda_projections import compute_lda_projection

fig, axes = plt.subplots(1, len(TERMS), figsize=(7 * len(TERMS), 6))
for ax, term in zip(np.atleast_1d(axes), TERMS):
    X, y = datasets[term]["X"], datasets[term]["y"]
    Z, _ = compute_lda_projection(X, y)
    palette = create_label_palette(len(np.unique(y)))
    ax.scatter(Z[:, 0], Z[:, 1], c=[palette[v] for v in y],
               s=12, alpha=0.7, linewidths=0)
    ax.set_title(f"{term} — LDA (supervised)"); ax.axis("off")
fig.tight_layout()
fig.savefig(FIGDIR / "lda_views.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Diachronic composition

No embedding required: the share of each predicted sense per century. A first, robust look at lineage before any projection.

In [ ]:
for term in TERMS:
    df = datasets[term]["df"].dropna(subset=["century"])
    if df.empty:
        print(f"{term}: no dated rows"); continue
    counts = df.groupby(["century", "sense"]).size().unstack(fill_value=0).sort_index()
    share = counts.div(counts.sum(axis=1), axis=0)
    ax = share.plot.area(figsize=(10, 5), cmap="tab10", alpha=0.85, linewidth=0)
    ax.set_title(f"{term}: predicted-sense composition by century")
    ax.set_xlabel("century midpoint (year)"); ax.set_ylabel("share")
    ax.legend(loc="upper left", fontsize=8, ncol=2, title="sense")
    plt.tight_layout()
    plt.savefig(FIGDIR / f"{term}_century_composition.png", dpi=150, bbox_inches="tight")
    plt.show()

## 8. Lineage — century centroids → trajectories

Project **all** points to 2D, then average coordinates per `(sense, century)` and
connect consecutive valid centuries with time on the Z axis. Set `LINEAGE_SPACE` to
`"lda"` for the sharp supervised view or `"umap"` for the label-free one.

In [ ]:
from projections_helpers import plot_3d_century_trajectories

LINEAGE_SPACE = "lda"   # "lda" (supervised) or "umap" (unsupervised)


def project_2d(term, space):
    X, y = datasets[term]["X"], datasets[term]["y"]
    if space == "lda":
        Z, _ = compute_lda_projection(X, y)
        return Z
    Xs, ys, idx = subsample(X, y)   # UMAP has no exact transform: project the subsample
    Zs = umap.UMAP(n_neighbors=15, min_dist=0.1, metric="cosine",
                   random_state=SEED).fit_transform(Xs)
    Z = np.full((len(X), 2), np.nan)
    Z[idx] = Zs
    return Z


def sense_names(term):
    _, _, lm = specs[term]
    if lm:
        return {int(k): v.replace("_", " ") for k, v in lm.items()}
    return {int(v): f"sense {v}" for v in sorted(datasets[term]["df"]["sense"].unique())}


def build_trajectories(df, Z2, min_samples=MIN_CENTURY_SAMPLES):
    d = df.assign(x2=Z2[:, 0], y2=Z2[:, 1]).dropna(subset=["x2", "century"])
    traj = {}
    for sense, g in d.groupby("sense"):
        rows = sorted(
            (int(c), float(gg["x2"].mean()), float(gg["y2"].mean()), len(gg))
            for c, gg in g.groupby("century")
        )
        traj[int(sense)] = {
            "centuries": [r[0] for r in rows],
            "x": [r[1] for r in rows],
            "y": [r[2] for r in rows],
            "z": [r[0] for r in rows],
            "sample_counts": [r[3] for r in rows],
            "marker_sizes": [50 + 100 * np.log10(max(r[3], 1)) for r in rows],
            "is_valid": [r[3] >= min_samples for r in rows],
        }
    return traj


for term in TERMS:
    df = datasets[term]["df"]
    Z2 = project_2d(term, LINEAGE_SPACE)
    traj = build_trajectories(df, Z2)
    names = sense_names(term)
    palette = create_label_palette(len(names))
    fig = plot_3d_century_trajectories(
        trajectory_data=traj, label_names=names, palette=palette,
        title=f"{term}: sense lineage in {LINEAGE_SPACE.upper()} space (time on Z)",
        interpolate_gaps=True, max_gap_centuries=2,
    )
    fig.savefig(FIGDIR / f"{term}_lineage_{LINEAGE_SPACE}.png", dpi=200, bbox_inches="tight")
    plt.show()

## Notes

- **Source of truth**: regenerate `*_embeddigings_full_classified.parquet` with
  `6_embedding-classification-applied.ipynb`, then re-run from §2.
- **LDA is supervised** (uses labels to pick axes). Use §4 to judge whether the
  label-free view (§5) is already meaningful; if it is, prefer it for lineage claims.
- Figures are written to `figures/` and kept out of the notebook to avoid a huge `.ipynb`.
- Prediction column is `class_<model>_embedding_pred`; its `_sense` companion holds names.